In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import keras_tuner as kt

In [ ]:
df = pd.read_csv("Churn_Modelling.csv")
df.head()


In [ ]:
X = df.drop('Exited', axis=1)
y = df['Exited']

In [ ]:
X = X.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [ ]:
label_encoder_gender = LabelEncoder()
X['Gender'] = label_encoder_gender.fit_transform(X['Gender'])

In [ ]:
ct = ColumnTransformer(
    transformers=[('geo', OneHotEncoder(drop='first'), ['Geography'])],
    remainder='passthrough'
)
X = ct.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
def build_model(hp):
    model = keras.Sequential()

    model.add(keras.layers.Input(shape=(X_train.shape[1],)))

    model.add(keras.layers.Dense(
        units=hp.Int('units_input', min_value=8, max_value=128, step=8),
        activation='relu'
    ))

    for i in range(hp.Int('num_hidden_layers', 1, 5)):
        model.add(keras.layers.Dense(
            units=hp.Int(f'units_{i}', min_value=8, max_value=128, step=8),
            activation='relu'
        ))

    model.add(keras.layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
tunner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='ann_tuning_new',
    project_name='churn_new'
)

In [ ]:
tuner.search(X_train, y_train, epochs=10, validation_split=0.2)

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best parameters found:")
print("Input units:", best_hps.get('units_input'))
print("Number of hidden layers:", best_hps.get('num_hidden_layers'))
print("Learning rate:", best_hps.get('learning_rate'))